# Exercise 2.5 — Scaling, transformations and feature engineering (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day2/day2-03b-scaling.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

## Overview

In this exercise, you will explore different scaling methods, apply transformations to numerical variables, and create new features through feature engineering—all essential steps in preparing data for downstream data analysis. 

## The scenario

We are continuing with the hospital admissions data to predict length of stay. Its
variables are measured in different units: age in years, blood pressure in
mmHg and C-reactive protein (CRP) in mg/L.

This is the same file as in Exercise 2.4. If you do not have it yet,
[**download `hospital_admissions.csv`**](data/hospital_admissions.csv) into your `data` folder.

CRP is a marker of inflammation. Most patients have relatively low values, but
some have very high measurements. This exercise looks at how different scaling
methods handle those values, how a transformation changes a distribution, and
how to create a clinically useful feature.

### 1. Scaling: three different approaches

Compare Min-Max, Standard and Robust scaling for age, systolic blood pressure
and CRP.

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

hospital = pd.read_csv("data/hospital_admissions.csv")

scale_cols = ["age", "systolic_BP", "CRP_mg_L"]

for name, scaler in [
    ("minmax", MinMaxScaler()),
    ("standard", StandardScaler()),
    ("robust", RobustScaler())
]:
    hospital[[f"{c}_{name}" for c in scale_cols]] = (
        scaler.fit_transform(hospital[scale_cols])
    )

Compare the range of values:

In [ ]:
print(hospital.filter(like="CRP").describe().round(3))

> **Why the standard deviation reads 1.002, not 1**
>
> `StandardScaler` divides by the standard deviation computed with $n$ in the
> denominator; `describe()` reports it with $n - 1$. On 250 patients the two
> differ by 0.2%. R's `scale()` uses $n - 1$ throughout, so the R tab reports a
> standard deviation of exactly 1 and a maximum of 8.3977 rather than 8.415.
> Neither is wrong.

Visualise CRP before and after scaling:

In [ ]:
cols = ["CRP_mg_L", "CRP_mg_L_minmax",
        "CRP_mg_L_standard", "CRP_mg_L_robust"]

hospital[cols].hist(bins=30, figsize=(9, 6))
plt.tight_layout()
plt.show()

> **Think about it**
>
> 1. What happens to the range after Min-Max scaling?
> 2. What happens to the mean and standard deviation after Standard scaling?
> 3. How does Robust scaling differ?
> 4. Does any scaling method eliminate CRP's skewness?

> **Worked answer**
>
> **Min-Max** puts every value between 0 and 1: the lowest CRP, 0.8, becomes 0
> and the highest, 264.6, becomes 1. Look at where the typical patient lands,
> though — the median is 0.037. Most of the cohort is crammed into the bottom
> tenth of the scale.
>
> **Standard** gives a mean of 0 and a standard deviation of 1. It is not
> bounded: the highest CRP sits about 8.4 standard deviations above the mean.
>
> **Robust** centres on the median instead of the mean, so the median becomes 0,
> and divides by the interquartile range instead of the standard deviation. The
> highest patient is 13.7 interquartile ranges out.
>
> **No.** Measure it rather than judging the histograms by eye:

In [ ]:
from scipy.stats import skew

print(hospital.filter(like="CRP").apply(skew).round(2))

> 4.43 in all four columns. Subtracting a constant and dividing by a constant
> cannot change a shape. The four histograms are the same picture with different
> numbers on the axis.

> **Key observation**
>
> All three methods change the numerical scale, but preserve the underlying shape
> of the distribution.

------------------------------------------------------------------------

### 2. Outliers: what happens to the other patients?

Imagine a patient has an exceptionally high CRP. How does that influence the
scaling of everyone else's measurements?

Compare the largest CRP values before and after Min-Max scaling:

In [ ]:
print(hospital["CRP_mg_L"].nlargest(5).tolist())

print(hospital["CRP_mg_L_minmax"].nlargest(5).round(3).tolist())

Now make the maximum CRP three times larger, change nothing else, and scale
again:

In [ ]:
crp_extreme = hospital["CRP_mg_L"].copy()
crp_extreme.loc[crp_extreme.idxmax()] *= 3

scaled_extreme = MinMaxScaler().fit_transform(
    crp_extreme.to_frame()
).ravel()

print(np.round(sorted(scaled_extreme, reverse=True)[:5], 3).tolist())

> **Think about it**
>
> 1. What happened to the scaled values of the other patients?
> 2. Which scaling method would be least affected by an extreme CRP measurement?
> 3. Should a clinically plausible extreme value automatically be removed?

> **Worked answer**
>
> **They all moved, though none of them changed.** The patient with the
> second-highest CRP, 197.7, was at 0.746 and is now at 0.248. Their measurement
> is exactly what it was. Min-Max divides by the range, the range is set by the
> single largest value, and so one patient decides where everybody else sits.
>
> **Robust**, and you can check it:

In [ ]:
for values in [hospital["CRP_mg_L"], crp_extreme]:
    z = RobustScaler().fit_transform(values.to_frame()).ravel()
    print(np.round(sorted(z, reverse=True)[:5], 2).tolist())

> Only the first number is different. The second-highest patient is at 10.12
> before and 10.12 after, because the median and the interquartile range are not
> moved by one extreme value. Standard scaling falls in between: the same patient
> drops from 6.1 to 3.2, since the standard deviation is inflated by the outlier.
>
> **No.** A CRP of 265 is not an error; in a real patient it would mean a severe
> infection, and probably the patient a model most needs to get right. Removing a value
> because it is inconvenient for a scaler throws away the cases that matter. Pick
> a scaler, or a transformation, that copes with them instead.

> **Key observation**
>
> Min-Max scaling is sensitive to extreme values. Robust scaling uses the median
> and interquartile range, which are far less sensitive to them.

------------------------------------------------------------------------

### 3. Transformation: can we reduce skewness?

Scaling does not change the shape of a distribution. A nonlinear transformation
can. Compare the original CRP distribution with its log transformation:

In [ ]:
hospital["log_CRP"] = np.log1p(hospital["CRP_mg_L"])

hospital[["CRP_mg_L", "log_CRP"]].hist(
    bins=30, figsize=(8, 3)
)
plt.tight_layout()
plt.show()

> **Think about it**
>
> 1. How does the shape change?
> 2. What happens to the distance between low and high CRP measurements?
> 3. Why use `log1p()` rather than `log()`?
>
> **Optional.** Repeat with `length_of_stay`, the eventual prediction target.
> Does its distribution also benefit from a log transformation?

> **Worked answer**
>
> **The long right tail is pulled in** and the distribution becomes close to
> symmetric:

In [ ]:
print({"CRP": round(skew(hospital["CRP_mg_L"]), 2),
       "log_CRP": round(skew(hospital["log_CRP"]), 2),
       "length_of_stay": round(skew(hospital["length_of_stay"]), 2),
       "log_length_of_stay": round(skew(np.log1p(hospital["length_of_stay"])), 2)})

print(np.round(np.log1p([5, 10, 100, 200]), 2).tolist())

> CRP's skewness falls from 4.43 to 0.38. Length of stay, the optional part,
> goes from 1.63 to 0.31 — it benefits too.
>
> **Distances stop being about units and start being about ratios.** On the
> original scale, going from 5 to 10 mg/L is a step of 5, and going from 100 to
> 200 is a step of 100. After the transformation those two doublings are almost
> the same size: about 0.61 and 0.69. That matches how CRP is read clinically —
> a doubling means something whether it starts low or high — and it stops the
> handful of very high values from dominating everything measured in distance.
>
> **Because the log of zero is minus infinity.** `log1p(x)` computes
> $\log(1 + x)$, so a value of 0 becomes 0 instead of breaking the column. In
> this file the lowest CRP is 0.8, so a plain `log()` would have run — but a
> laboratory can report a result below its detection limit as 0, and `log1p()`
> is the habit that survives that.

------------------------------------------------------------------------

### 4. Feature engineering: calculate BMI

The dataset includes height and weight, but not BMI. Create it and look at its
distribution:

In [ ]:
hospital["BMI"] = (
    hospital["weight_kg"] /
    (hospital["height_cm"] / 100) ** 2
)

hospital["BMI"].hist(bins=25, figsize=(7, 2.8))
plt.xlabel("BMI (kg/m²)")
plt.tight_layout()
plt.show()

> **Think about it**
>
> 1. Why might BMI be more informative than weight alone?
> 2. Is BMI adding new measurements, or combining existing information?
> 3. What other clinically meaningful features could you derive from the
>    variables you already have?

> **Worked answer**
>
> **Because weight means nothing without height.** Two patients from this file:

In [ ]:
print(hospital.loc[hospital["patient_id"].isin(["H134", "H184"]),
                   ["patient_id", "height_cm", "weight_kg", "BMI"]]
      .round(1).to_string(index=False))

> Both weigh about 79 kg. One is 142 cm tall and has a BMI of 39.0; the other is
> 201 cm and has a BMI of 19.6. On weight alone a model sees two identical
> patients. On BMI it sees one who is obese and one at the lean end of normal.
>
> **Combining.** No new measurement was taken; BMI is arithmetic on two columns
> that were already there. What it adds is the *relationship* between them, in
> the form a clinician would use. A model handed height and weight separately
> would have to discover that ratio for itself, and a linear model cannot.
>
> **Others from this table:** the shock index, heart rate divided by systolic
> blood pressure, which is used at the bedside to flag circulatory compromise; a
> flag for tachycardia (heart rate over 100) or hypotension (systolic below 90);
> age bands; the log of CRP you have just made. The test is the same each time:
> does the new column say something clinically meaningful that the raw columns
> only imply?

------------------------------------------------------------------------

### What to take from this

- **Scaling changes the numerical range of a variable, not its shape.** CRP was
  as skewed after all three scalers as before.
- **Robust scaling is less sensitive to extreme values.** One inflated
  measurement moved every other patient under Min-Max and none under Robust.
- **Transformation can reduce skewness and changes the relationships between
  values.** A log turned steps of units into steps of ratios.
- **Feature engineering creates new variables from existing information.** BMI
  added no measurement and still separated two patients that weight could not.

> **Remember**
>
> Every minimum, maximum, mean, standard deviation, median and interquartile
> range in this exercise was computed from all 250 patients. When you build a
> predictive model, anything learned from the data — including a scaler — must be
> fitted on the training set only, not the full dataset.

Next: Exercise 2.6 combines imputation, categorical encoding, scaling
and feature engineering into one preprocessing pipeline.